In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist.silver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Origem: olist.bronze.tb_customers
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_customers = spark.table("olist.bronze.tb_customers")

# Window Function particionada por customer_id, ordenada pelo timestamp_ingestion
# mais recente para garantir que apenas o registro mais recente seja mantido
window_dedup = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_consumidores = (
    df_customers
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.col("customer_unique_id").alias("id_consumidor_unico"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado"),
    )
)

(
    df_dim_consumidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_consumidores")
)

print(f"✅ olist.silver.dim_consumidores ({df_dim_consumidores.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_products
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_products = spark.table("olist.bronze.tb_products")

window_dedup = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_produtos = (
    df_products
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_name_lenght").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").alias("tamanho_descricao_produto"),
        F.col("product_photos_qty").alias("quantidade_fotos"),
        F.col("product_weight_g").alias("peso_produto_gramas"),
        F.col("product_length_cm").alias("comprimento_centimetros"),
        F.col("product_height_cm").alias("altura_centimetros"),
        F.col("product_width_cm").alias("largura_centimetros"),
    )
)

(
    df_dim_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_produtos")
)

print(f"✅ olist.silver.dim_produtos ({df_dim_produtos.count()} registros)")

In [0]:
# Origem: olist.bronze.tb_sellers
# Deduplicação via Window Function pelo timestamp_ingestion mais recente
df_sellers = spark.table("olist.bronze.tb_sellers")

window_dedup = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

df_dim_vendedores = (
    df_sellers
    .withColumn("rank", F.row_number().over(window_dedup))
    .filter(F.col("rank") == 1)
    .drop("rank", "timestamp_ingestion")
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado"),
    )
)

(
    df_dim_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.silver.dim_vendedores")
)

print(f"✅ olist.silver.dim_vendedores ({df_dim_vendedores.count()} registros)")